In [3]:

import numpy as np
import pandas as pd

train_model_data= pd.read_csv('../data/processed/task7/train_model_data.csv')
eval_df = pd.read_csv('../data/processed/task7/eval_df.csv')

print("Train shape:", train_model_data.shape)
print("Validation shape:", eval_df.shape)



Train shape: (5048, 71)
Validation shape: (494, 77)


In [5]:
# 1. Build price references from TRAINING data only
train_ref = train_model_data[(train_model_data['listing_type'] == 'phone')& train_model_data['price'].notna()].copy()


In [45]:
# Fallback thresholds (initial values for experiments)
MIN_MODEL_COUNT = 3
MIN_FAMILY_COUNT = 10
MIN_BRAND_COUNT = 20

Different levels 

In [46]:
# Global fallback
global_median = train_ref['price'].median()

# 2. Model-level reference
model_ref = train_ref[train_ref['model_key_norm'].notna()& train_ref['model_key_norm'].ne('unknown')]
model_stats = model_ref.groupby('model_key_norm')['price'].agg(
    model_median='median',model_count='size'
)

# 3. Family-level reference
family_ref = train_ref[train_ref['family'].notna()]
family_stats = family_ref.groupby('family')['price'].agg(
    family_median='median',
    family_count='size'
)

# 4. Brand-level reference
brand_ref = train_ref[train_ref['brand'].notna()]
brand_stats = brand_ref.groupby('brand')['price'].agg(
    brand_median='median',
    brand_count='size'
)


In [47]:
def hierarchical_fallback(row):

    model = row['model_key_norm']
    family = row['family']
    brand = row['brand']

    # 1. Exact model
    if model in model_stats.index and model != 'unknown':
        stats = model_stats.loc[model]
        if stats['model_count'] >= MIN_MODEL_COUNT:
            return stats['model_median'], 'model'

    # 2. Family
    if family in family_stats.index:
        stats = family_stats.loc[family]
        if stats['family_count'] >= MIN_FAMILY_COUNT:
            return stats['family_median'], 'family'

    # 3. Brand
    if brand in brand_stats.index:
        stats = brand_stats.loc[brand]
        if stats['brand_count'] >= MIN_BRAND_COUNT:
            return stats['brand_median'], 'brand'

    # 4. Global fallback
    return global_median, 'global'

In [48]:
# Evaluate on phone listings with known prices
fallback_eval = eval_df[(eval_df['listing_type'] == 'phone')& eval_df['price'].notna()& eval_df['model_key_norm'].notna()].copy()

# Apply fallback
fallback_results = fallback_eval.apply(hierarchical_fallback,axis=1)


if isinstance(fallback_results, pd.DataFrame):
    fallback_results.columns = [
        'fallback_price', 'fallback_level'
    ]
else:
    fallback_results = pd.DataFrame(
        fallback_results.tolist(),
        index=fallback_eval.index,
        columns=['fallback_price', 'fallback_level']
    )

fallback_eval = pd.concat(
    [fallback_eval, fallback_results],
    axis=1
)

In [49]:
#  Calculate errors
fallback_eval['fallback_absolute_error'] = ( fallback_eval['price'] - fallback_eval['fallback_price']).abs()

print(
    "Fallback MAE:",
    round(fallback_eval['fallback_absolute_error'].mean(), 2)
)

print("\nFallback level usage:")
display(
    fallback_eval['fallback_level']
    .value_counts()
    .rename_axis('fallback_level')
    .reset_index(name='ads')
)




Fallback MAE: 6662.38

Fallback level usage:


,fallback_level,ads
0,model,373
1,family,115
2,brand,5
3,global,1


In [50]:
# 8. Evaluate each level separately
fallback_summary = (fallback_eval.groupby('fallback_level').agg(
        ads=('fallback_absolute_error', 'size'),
        MAE=('fallback_absolute_error', 'mean'),
        median_price=('price', 'median')
    )
    .reset_index()
)

display(fallback_summary.round(2))

,fallback_level,ads,MAE,median_price
0,brand,5,6580.00,12500.0
1,family,115,9886.51,9000.0
2,global,1,30043.00,45543.0
3,model,373,5606.77,23000.0
